# 02 — Metrics (ERR-6) + Dashboard Export

This notebook computes early diffusion metrics for selected BLACKPINK anchor tracks using Spotify daily chart data (Day0–Day6).

Metric:
- ERR-6 = Day6 / Peak within first 7 days

Outputs (exported to dashboard/data):
- metrics_with_era.csv
- era_market_anchor_summary.csv
- main_vs_pre_gap.csv

In [10]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

# ---- Project paths ----
PROJECT_DIR = Path("/Users/apple/Desktop/BLACKPINK_global_spread_project")
DATA_DIR = PROJECT_DIR / "data"
CHARTS_DIR = DATA_DIR / "spotify_charts"
DASHBOARD_DATA_DIR = PROJECT_DIR / "dashboard" / "data"

ANCHOR_CONFIG_PATH = DATA_DIR / "group_release_anchor_config.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("CHARTS_DIR exists:", CHARTS_DIR.exists(), CHARTS_DIR)
print("ANCHOR_CONFIG exists:", ANCHOR_CONFIG_PATH.exists(), ANCHOR_CONFIG_PATH)
print("DASHBOARD_DATA_DIR:", DASHBOARD_DATA_DIR)
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR: /Users/apple/Desktop/BLACKPINK_global_spread_project
CHARTS_DIR exists: True /Users/apple/Desktop/BLACKPINK_global_spread_project/data/spotify_charts
ANCHOR_CONFIG exists: True /Users/apple/Desktop/BLACKPINK_global_spread_project/data/group_release_anchor_config.csv
DASHBOARD_DATA_DIR: /Users/apple/Desktop/BLACKPINK_global_spread_project/dashboard/data


In [11]:
anchor = pd.read_csv(ANCHOR_CONFIG_PATH)
print("anchor_config columns:", anchor.columns.tolist())
display(anchor.head(5))

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# 必需：uri（或 track_id）
uri_col = find_col(anchor, ["uri", "track_uri", "spotify_uri", "URI"])
track_id_col = find_col(anchor, ["track_id", "Track ID", "TrackID", "spotify_track_id"])

# 可选：track_name / anchor_type / era / market / release_date
name_col = find_col(anchor, ["track_name", "Track Name", "title", "song", "track"])
anchor_type_col = find_col(anchor, ["anchor_type", "Anchor Type", "type"])
era_col = find_col(anchor, ["era", "Era"])
market_col = find_col(anchor, ["market", "Market"])
release_date_col = find_col(anchor, ["release_date", "Release Date", "releaseDate", "date"])

if uri_col is None and track_id_col is None:
    raise KeyError("anchor_config 必须至少包含 uri 或 track_id 其中之一。")

# 统一生成 uri
if uri_col is None:
    # 只有 track_id 的话，拼成 spotify uri
    anchor["uri"] = anchor[track_id_col].astype(str).str.strip().apply(lambda x: f"spotify:track:{x}")
else:
    anchor["uri"] = anchor[uri_col].astype(str).str.strip()

# 统一 track_id
anchor["track_id"] = anchor["uri"].str.extract(r"spotify:track:([A-Za-z0-9]+)", expand=False)

# 统一 track_name（没有也不致命）
anchor["track_name"] = anchor[name_col].astype(str) if name_col else ""

# 统一 anchor_type / era / market
anchor["anchor_type"] = anchor[anchor_type_col].astype(str) if anchor_type_col else "Main"
anchor["era"] = anchor[era_col].astype(str) if era_col else "Unknown"

# market 允许不存在：不存在就默认三市场都算
DEFAULT_MARKETS = ["Global", "US", "ID"]
if market_col:
    anchor["market"] = anchor[market_col].astype(str)
else:
    anchor["market"] = "ALL"

# release_date 必须有（否则 Day0-Day6 没法定义）
if release_date_col is None:
    raise KeyError("anchor_config 必须包含 release_date（用于定义 Day0-Day6）。")

anchor["release_date"] = pd.to_datetime(anchor[release_date_col], errors="coerce")
if anchor["release_date"].isna().any():
    bad = anchor[anchor["release_date"].isna()]
    raise ValueError(f"anchor_config 有无法解析的 release_date：\n{bad.head(10)}")

display(anchor.head(10))
print("unique markets in anchor:", anchor["market"].unique().tolist())

anchor_config columns: ['track_id', 'track_name', 'release_date', 'anchor_type', 'anchor_reason']


,track_id,track_name,release_date,anchor_type,anchor_reason
0,BP03,As If It’s Your Last,2017/6/22,Main,Standalone single anchoring pre-album era
1,BP04,DDU-DU DDU-DU,2018/6/15,Main,Square Up global breakout anchor
2,BP05,Kill This Love,2019/4/5,Main,EP anchor tied to Coachella-era exposure
3,BP06,How You Like That,2020/6/26,Pre,Album pre-release signal driving initial momentum
4,BP07,Lovesick Girls,2020/10/2,Main,The Album official title-track anchor


,track_id,track_name,release_date,anchor_type,anchor_reason,uri,era,market
0,BP03,As If It’s Your Last,2017-06-22,Main,Standalone single anchoring pre-album era,spotify:track:BP03,Unknown,ALL
1,BP04,DDU-DU DDU-DU,2018-06-15,Main,Square Up global breakout anchor,spotify:track:BP04,Unknown,ALL
2,BP05,Kill This Love,2019-04-05,Main,EP anchor tied to Coachella-era exposure,spotify:track:BP05,Unknown,ALL
3,BP06,How You Like That,2020-06-26,Pre,Album pre-release signal driving initial momentum,spotify:track:BP06,Unknown,ALL
4,BP07,Lovesick Girls,2020-10-02,Main,The Album official title-track anchor,spotify:track:BP07,Unknown,ALL
5,BP08,Pink Venom,2022-08-19,Pre,Born Pink pre-release signal for global attent...,spotify:track:BP08,Unknown,ALL
6,BP09,Shut Down,2022-09-16,Main,Born Pink album anchor with synchronized globa...,spotify:track:BP09,Unknown,ALL
7,BP10,Jump,2025-07-11,Main,Digital single with live premiere to release t...,spotify:track:BP10,Unknown,ALL


unique markets in anchor: ['ALL']


In [12]:
# 如果 market=ALL，把每条 anchor 复制到三市场
if (anchor["market"] == "ALL").any():
    base = anchor[anchor["market"] == "ALL"].copy()
    others = anchor[anchor["market"] != "ALL"].copy()

    expanded = []
    for m in ["Global", "US", "ID"]:
        temp = base.copy()
        temp["market"] = m
        expanded.append(temp)

    anchor = pd.concat([others] + expanded, ignore_index=True)

# 只保留三大市场（你 dashboard 就是 Global/US/ID）
market_map = {
    "global": "Global",
    "us": "US",
    "united states": "US",
    "indonesia": "ID",
    "id": "ID",
}
anchor["market"] = anchor["market"].astype(str).str.strip()
anchor["market_norm"] = anchor["market"].str.lower().map(market_map).fillna(anchor["market"])

anchor = anchor[anchor["market_norm"].isin(["Global", "US", "ID"])].copy()
anchor["market"] = anchor["market_norm"]
anchor = anchor.drop(columns=["market_norm"])

print("anchor rows:", len(anchor))
print("anchor markets:", anchor["market"].unique().tolist())
print("anchor uris:", anchor["uri"].nunique())

anchor rows: 24
anchor markets: ['Global', 'US', 'ID']
anchor uris: 8


In [13]:
# 文件名例子：regional-global-daily-2017-06-22.csv
pat = re.compile(r"regional-(?P<region>.+?)-daily-(?P<date>\d{4}-\d{2}-\d{2})\.csv$", re.I)

def parse_market_and_date(path: Path):
    m = pat.search(path.name)
    if not m:
        return None, None
    region = m.group("region").lower().strip()
    date = pd.to_datetime(m.group("date"))
    # region -> market
    if region == "global":
        market = "Global"
    elif region == "us":
        market = "US"
    elif region in ["id", "indonesia"]:
        market = "ID"
    else:
        return None, None
    return market, date

# 只读 Day0-Day6 可能涉及的日期范围（全量读也行，但这更稳更快）
min_d = anchor["release_date"].min()
max_d = anchor["release_date"].max() + pd.Timedelta(days=6)

files = []
for f in sorted(CHARTS_DIR.glob("*.csv")):
    market, d = parse_market_and_date(f)
    if market is None or d is None:
        continue
    if min_d <= d <= max_d:
        files.append((f, market, d))

print("files in range:", len(files), "from", min_d.date(), "to", max_d.date())
print("sample:", files[0] if files else "NONE")

anchor_uris = set(anchor["uri"].unique().tolist())

rows = []
usecols = ["uri", "artist_names", "track_name", "streams"]

for (fpath, market, chart_date) in files:
    df_day = pd.read_csv(fpath, usecols=usecols)
    df_day = df_day[df_day["uri"].isin(anchor_uris)].copy()
    if df_day.empty:
        continue
    df_day["market"] = market
    df_day["chart_date"] = chart_date
    rows.append(df_day)

charts = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=usecols + ["market","chart_date"])
charts["chart_date"] = pd.to_datetime(charts["chart_date"])
charts["streams"] = pd.to_numeric(charts["streams"], errors="coerce")

print("charts rows:", len(charts))
display(charts.head(10))

files in range: 168 from 2017-06-22 to 2025-07-17
sample: (PosixPath('/Users/apple/Desktop/BLACKPINK_global_spread_project/data/spotify_charts/regional-global-daily-2017-06-22.csv'), 'Global', Timestamp('2017-06-22 00:00:00'))
charts rows: 0


,uri,artist_names,track_name,streams,market,chart_date


In [14]:
# 合并（按 uri + market）
merged = charts.merge(
    anchor[["uri", "track_id", "track_name", "anchor_type", "era", "market", "release_date"]],
    on=["uri", "market"],
    how="inner",
    suffixes=("", "_anchor")
)

merged["day_index"] = (merged["chart_date"] - merged["release_date"]).dt.days
merged = merged[(merged["day_index"] >= 0) & (merged["day_index"] <= 6)].copy()

# 如果 track_name 在 charts 里更可靠，你可以用 charts 的 track_name
# 这里保留 charts 的 track_name，同时把 anchor_config 的 name 作为备用
merged["track_name_anchor"] = merged["track_name_anchor"].astype(str)
merged["track_name"] = merged["track_name"].fillna("").astype(str)
merged.loc[merged["track_name"].eq(""), "track_name"] = merged.loc[merged["track_name"].eq(""), "track_name_anchor"]

print("merged rows (Day0-6):", len(merged))
display(merged.sort_values(["market","uri","chart_date"]).head(20))

merged rows (Day0-6): 0


,uri,artist_names,track_name,streams,market,chart_date,track_id,track_name_anchor,anchor_type,era,release_date,day_index


In [15]:
# peak within day0-6
peak = (
    merged.groupby(["track_id", "uri", "track_name", "market", "anchor_type", "era"], as_index=False)
    .agg(peak_streams_calc=("streams", "max"))
)

# day6 streams
day6 = merged[merged["day_index"] == 6].copy()
day6 = (
    day6.groupby(["track_id", "uri", "track_name", "market", "anchor_type", "era"], as_index=False)
    .agg(day6_streams=("streams", "max"))
)

metrics = peak.merge(day6, on=["track_id","uri","track_name","market","anchor_type","era"], how="left")
metrics["retention_day6_vs_peak"] = metrics["day6_streams"] / metrics["peak_streams_calc"]

# 可选：补一列 release_date（方便 trace）
release_map = anchor.drop_duplicates(["uri","market"])[["uri","market","release_date"]]
metrics = metrics.merge(release_map, on=["uri","market"], how="left")

# 清理
metrics["retention_day6_vs_peak"] = metrics["retention_day6_vs_peak"].replace([np.inf, -np.inf], np.nan)

print("metrics rows:", len(metrics))
display(metrics.head(20))

metrics rows: 0


,track_id,uri,track_name,market,anchor_type,era,peak_streams_calc,day6_streams,retention_day6_vs_peak,release_date


In [16]:
# 1) metrics_with_era.csv
metrics_out = metrics.copy()

# 2) era_market_anchor_summary.csv
era_market_anchor_summary = (
    metrics_out.groupby(["era","market","anchor_type"], as_index=False)
    .agg(
        n_tracks=("track_id","nunique"),
        avg_err6=("retention_day6_vs_peak","mean"),
        median_err6=("retention_day6_vs_peak","median"),
    )
)

# 3) main_vs_pre_gap.csv (avg ERR-6 main - pre)
pivot_gap = (
    era_market_anchor_summary.pivot_table(
        index=["era","market"],
        columns="anchor_type",
        values="avg_err6",
        aggfunc="mean"
    )
    .reset_index()
)

# 兼容：如果没有 Pre 或 Main，gap 会变 NaN
pivot_gap["main_minus_pre"] = pivot_gap.get("Main", np.nan) - pivot_gap.get("Pre", np.nan)
main_vs_pre_gap = pivot_gap[["era","market","main_minus_pre"]].copy()

# ---- Export ----
metrics_path = DASHBOARD_DATA_DIR / "metrics_with_era.csv"
era_summary_path = DASHBOARD_DATA_DIR / "era_market_anchor_summary.csv"
gap_path = DASHBOARD_DATA_DIR / "main_vs_pre_gap.csv"

metrics_out.to_csv(metrics_path, index=False)
era_market_anchor_summary.to_csv(era_summary_path, index=False)
main_vs_pre_gap.to_csv(gap_path, index=False)

print("✅ Dashboard data exported:")
print([metrics_path.name, era_summary_path.name, gap_path.name])
print("->", DASHBOARD_DATA_DIR)

✅ Dashboard data exported:
['metrics_with_era.csv', 'era_market_anchor_summary.csv', 'main_vs_pre_gap.csv']
-> /Users/apple/Desktop/BLACKPINK_global_spread_project/dashboard/data
